# Studio 2 — Manual Cell-Type Annotation and CellTypist Comparison

**Run this in Google Colab:** download this `.ipynb`, then choose **File → Upload notebook** in Colab. It is intentionally a *guided analysis*, not an answer key. The notebook gives you the reproducible setup through clustering. After that, you ask an AI coach for **one code cell or one figure at a time** and do not proceed until you have inspected the result.

## Biological question

How do cluster-specific gene-expression patterns support defensible manual cell-type labels, and where do those labels agree or disagree with a pretrained CellTypist reference model?

## Learning outcomes

By the end of the studio, you will be able to:

1. preprocess raw single-cell counts and construct a baseline Leiden-clustered UMAP with Scanpy;
2. compare Leiden resolutions and select an annotation granularity using visual and marker-evidence criteria;
3. use cluster marker genes and curated marker panels to propose and document manual labels;
4. run a tissue-appropriate CellTypist pretrained model with its **majority-voting** option using the selected annotation clusters as local groups; and
5. compare manual and automated labels, retain agreement as a working consensus, and flag disagreement for review rather than forcing a label.

> **Important scientific boundary.** A cluster, UMAP position, marker list, or pretrained prediction is evidence for a label; none is ground truth by itself. Cell-level differential-expression p-values should not be interpreted as biological replicate-level inference.

## Prerequisite map

```text
Raw counts + gene symbols
          ↓
QC metrics → permissive filtering → log1p CP10k normalization
          ↓
HVG selection → PCA → neighbor graph → UMAP + baseline Leiden clusters
          ↓
Leiden resolution comparison → selected annotation cluster key
          ↓
cluster markers + curated marker panels → manual labels
          ↓
CellTypist label transfer + CellTypist majority voting
          ↓
manual vs automated comparison → agreement / Review adjudication
```

**Before you begin:** You need basic Python, a basic understanding of gene-expression counts, and permission to use an AI assistant. Use only the included public datasets or other non-sensitive, approved data. Do **not** paste protected, identifiable, unpublished, embargoed, or access-controlled data into an AI system.

## Public data choices

- `pbmc`: Scanpy’s PBMC 3k dataset, a healthy-donor 10x dataset. Use CellTypist’s `Immune_All_Low.pkl` model.
- `pancreas`: the scIB human pancreas benchmark dataset. Use CellTypist’s `Adult_Human_PancreaticIslet.pkl` model only after verifying that the query is human pancreatic-islet-like and that its 12-class reference scope fits your question.

The pancreas download is materially larger than PBMC and may take a few minutes.


## Studio workflow and deliverables

| Phase | Your decision | Evidence artifact |
|---|---|---|
| 1. Baseline | Which cells/genes and graph settings form a reasonable starting analysis? | `adata` with QC metadata, PCA, UMAP, and baseline `leiden` |
| 2. Resolution decision | What cluster granularity is biologically useful without unjustified splitting? | Resolution summary, one comparison UMAP, and `annotation_cluster` |
| 3. Manual annotation | Which labels are supported by **multiple** markers for each selected cluster? | Cluster-marker table, marker plots, and a cited cluster-to-label map |
| 4. Automated annotation | Is the CellTypist model appropriate for this organism/tissue and what does it predict? | Model scope note and CellTypist result columns |
| 5. Adjudication | Which labels agree, and which require review? | Normalized cross-tabulation and `consensus_label` |

**Required submission artifacts:** a resolution-comparison UMAP and rationale, a screenshot/export of the final UMAPs, your marker-evidence table, the manual-vs-CellTypist comparison, a short model-scope note, and a brief statement of one limitation or unresolved label.

**Do not tune parameters to obtain a preferred biological answer.** Record any changed parameter, why you changed it, and what diagnostic changed.


In [ ]:
# Install the Colab dependencies. The notebook records the resolved versions below.
%pip install --quiet "scanpy>=1.10,<1.12" "celltypist>=1.7" "leidenalg>=0.10" "seaborn>=0.13"

# If Colab asks to restart the runtime after installation, do so, then continue at the next cell.


In [ ]:
# Imports, reproducibility settings, and version record
import os
import random
import sys
import urllib.request
from pathlib import Path

import anndata as ad
import celltypist
from celltypist import models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

SEED = 17
random.seed(SEED)
np.random.seed(SEED)

sc.settings.set_figure_params(dpi=90, facecolor="white", frameon=False)
sc.settings.verbosity = 2

print("Python:", sys.version.split()[0])
print("Scanpy:", sc.__version__)
print("AnnData:", ad.__version__)
print("CellTypist:", celltypist.__version__)
print("Working directory:", os.getcwd())


## 1. Select the public dataset

Use `pbmc` for the fastest first pass. The `pancreas` option is a human pancreas benchmark with a `counts` layer. The loader deliberately restores its raw counts to `.X` before this notebook’s preprocessing starts.

Do not switch dataset midway through a run. Restart from this cell if you change it.


In [ ]:
# Choose exactly one public dataset for this run.
DATASET = "pbmc"  # allowed values: "pbmc" or "pancreas"

PANCREAS_URL = "https://api.figshare.com/v2/file/download/24539828"
PANCREAS_PATH = Path("/content/pancreas_scib.h5ad")

MODEL_BY_DATASET = {
    "pbmc": "Immune_All_Low.pkl",
    "pancreas": "Adult_Human_PancreaticIslet.pkl",
}

if DATASET == "pbmc":
    adata = sc.datasets.pbmc3k()
    dataset_source = "Scanpy pbmc3k: 2,700 PBMCs from a healthy donor"
elif DATASET == "pancreas":
    if not PANCREAS_PATH.exists():
        print("Downloading pancreas AnnData; this may take a few minutes...")
        urllib.request.urlretrieve(PANCREAS_URL, PANCREAS_PATH)
    adata = sc.read_h5ad(PANCREAS_PATH)
    if "counts" not in adata.layers:
        raise ValueError("The documented pancreas dataset should contain a raw-counts layer named 'counts'.")
    adata.X = adata.layers["counts"].copy()
    dataset_source = "scIB human pancreas benchmark, Figshare file 24539828"
else:
    raise ValueError("DATASET must be either 'pbmc' or 'pancreas'.")

adata.var_names_make_unique()
adata.obs_names_make_unique()
MODEL_NAME = MODEL_BY_DATASET[DATASET]

print(dataset_source)
print("Selected CellTypist model:", MODEL_NAME)
adata


## 2. Fixed, inspectable preprocessing baseline

This baseline is intentionally conservative and transparent. It calculates QC metrics and applies only a basic, documented filter (`min_genes` and `min_cells`). It does **not** silently remove cells by mitochondrial percentage or run doublet detection. Inspect those diagnostics before deciding whether an additional, dataset- and sample-aware filter is justified.

The code preserves a log1p, CP10k-normalized all-gene matrix in `adata.raw` **before** selecting highly variable genes. That is essential because CellTypist expects log1p-normalized data with 10,000 counts per cell and benefits from all matching genes.

For data with multiple biological samples, QC thresholds and any doublet analysis should be examined per sample. This public pancreas data includes a technology field (`tech`), not independent biological replicate metadata for inference.


In [ ]:
# Baseline parameters — record any change in your analysis notes.
MIN_GENES = 200
MIN_CELLS = 3
TARGET_SUM = 10_000
N_TOP_HVGS = 2_000
N_PCS_REQUESTED = 30
N_NEIGHBORS = 15
LEIDEN_RESOLUTION = 0.7

# QC metrics are calculated from raw counts.
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=True,
    inplace=True,
)

print("Before filtering:", adata.n_obs, "cells ×", adata.n_vars, "genes")
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe().round(2))

# Basic filtering only. Do not add a mitochondrial cutoff without inspecting a QC diagnostic.
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print("After basic filtering:", adata.n_obs, "cells ×", adata.n_vars, "genes")

# Preserve counts and construct CellTypist-compatible log1p CP10k values.
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)
adata.raw = adata  # all filtered genes, log1p CP10k; do not overwrite later

# Select features for graph construction, then construct PCA, neighbors, UMAP, and clusters.
batch_key = "tech" if DATASET == "pancreas" and "tech" in adata.obs.columns else None
hvg_kwargs = dict(n_top_genes=N_TOP_HVGS, flavor="seurat")
if batch_key is not None:
    hvg_kwargs["batch_key"] = batch_key
sc.pp.highly_variable_genes(adata, **hvg_kwargs)

if int(adata.var["highly_variable"].sum()) < 2:
    raise ValueError("Too few highly variable genes. Recheck the input matrix and QC choices.")

adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
N_PCS = min(N_PCS_REQUESTED, adata.n_obs - 1, adata.n_vars - 1)
if N_PCS < 2:
    raise ValueError("Too few cells or HVGs remain for PCA.")

sc.tl.pca(adata, n_comps=N_PCS, svd_solver="arpack", random_state=SEED)
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.tl.leiden(
    adata,
    resolution=LEIDEN_RESOLUTION,
    key_added="leiden",
    random_state=SEED,
)

print("Analysis object ready:")
print(adata)
print("Clusters:", adata.obs["leiden"].value_counts().sort_index().to_dict())


## 3. Leiden-resolution sensitivity and annotation granularity

The neighbor graph and UMAP are fixed in the baseline. This section computes several **partitions of that same graph**. A larger Leiden resolution usually yields more clusters, but more clusters are not automatically more biological detail. Conversely, a lower resolution can merge identities that show distinct marker programs.

Run the code cell to create the candidate cluster columns and a size summary. Then use **STAGE 0A** to create one visual comparison. Choose an `ANNOTATION_CLUSTER_KEY` only after you have inspected the resolution figure and documented a rationale. A defensible choice:

- separates groups with coherent, distinct multi-gene marker programs;
- does not create small clusters supported only by mitochondrial/ribosomal expression, likely doublets, or a single gene;
- avoids merging groups with contradictory lineage markers; and
- is reported as an analytical choice, not as proof of a fixed number of cell types.

You will annotate only `adata.obs["annotation_cluster"]` in the rest of the studio. The baseline `adata.obs["leiden"]` and all candidate columns remain intact for comparison.

In [ ]:
# Compare Leiden partitions of the existing neighbor graph. This does not rerun PCA, neighbors, or UMAP.
LEIDEN_RESOLUTIONS = [0.30, 0.50, 0.70, 1.00]
RESOLUTION_KEY_BY_VALUE = {}
resolution_records = []

for resolution in LEIDEN_RESOLUTIONS:
    key = f"leiden_res_{resolution:.2f}".replace(".", "_")
    sc.tl.leiden(adata, resolution=resolution, key_added=key, random_state=SEED)
    RESOLUTION_KEY_BY_VALUE[resolution] = key
    cluster_sizes = adata.obs[key].value_counts()
    resolution_records.append({
        "resolution": resolution,
        "cluster_key": key,
        "n_clusters": int(cluster_sizes.size),
        "smallest_cluster_cells": int(cluster_sizes.min()),
        "median_cluster_cells": float(cluster_sizes.median()),
        "largest_cluster_cells": int(cluster_sizes.max()),
    })

resolution_summary = pd.DataFrame(resolution_records).sort_values("resolution")
display(resolution_summary)

# After STAGE 0A/0B, edit this to the selected candidate and rerun this cell.
# The 0.70 candidate is the documented baseline default, not a pre-judged best choice.
ANNOTATION_CLUSTER_KEY = RESOLUTION_KEY_BY_VALUE[LEIDEN_RESOLUTION]
if ANNOTATION_CLUSTER_KEY not in adata.obs.columns:
    raise KeyError(f"Unknown cluster key: {ANNOTATION_CLUSTER_KEY}")

adata.obs["annotation_cluster"] = adata.obs[ANNOTATION_CLUSTER_KEY].astype("category")
print("Selected annotation cluster key:", ANNOTATION_CLUSTER_KEY)
print("Selected cluster sizes:", adata.obs["annotation_cluster"].value_counts().sort_index().to_dict())

## 4. Data contract for the AI-coached phase

Every later AI request must honor this contract. If a generated code cell violates it, reject it and ask for a corrected, minimal cell.

```text
Canonical object:                 adata
Baseline cluster key:             adata.obs["leiden"]
Candidate resolution keys:        RESOLUTION_KEY_BY_VALUE and adata.obs["leiden_res_…"]
Selected annotation cluster key:  ANNOTATION_CLUSTER_KEY
Cluster labels for all later work: adata.obs["annotation_cluster"]
Embedding:                        adata.obsm["X_umap"]
Analysis matrix:                  adata.X = scaled highly-variable genes
Marker / CellTypist matrix:       adata.raw.X = log1p CP10k values for all filtered genes
Original filtered counts:         adata.layers["counts"] (HVG subset only after feature selection)
Dataset choice:                   DATASET
Selected model:                   MODEL_NAME
Random seed:                      SEED
```

**Do not** overwrite `adata.raw`, replace `adata.X` with an untracked matrix, regenerate clusters silently, use hidden reference labels if present in the source dataset, or claim that a plot proves cell identity. Generated figure code must begin with `# STAGE n` and save the artifact to `outputs/` as both a PNG and a PDF when practical.

Create the output directory now.

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Figures and tables will be written to:", OUTPUT_DIR.resolve())


## 5. Marker panels for the manual-annotation prompt

These are **starting panels**, not a label key. A valid manual assignment needs coherent expression of multiple positive markers, compatibility with the tissue and organism, and consideration of conflicting markers, doublets, low quality, ambient RNA, and cell state. Never assign a label from one marker alone.

The PBMC panel emphasizes broad lineages. The pancreas panel includes major endocrine and non-endocrine populations. It does not assert that every class exists in this dataset.


In [ ]:
# Marker panels supplied as context to the AI coach. Do not edit these after inspecting results
# merely to make a preferred label appear stronger; record any justified addition.
MARKERS = {
    "pbmc": {
        "T_cells": ["CD3D", "CD3E", "TRBC2"],
        "CD4_memory_or_naive_T": ["IL7R", "CCR7", "LTB", "MALAT1"],
        "Cytotoxic_T_or_NK": ["NKG7", "PRF1", "GZMB", "GNLY"],
        "B_cells": ["MS4A1", "CD79A", "CD74", "HLA-DRA"],
        "CD14_monocytes": ["LYZ", "S100A8", "S100A9", "CTSD"],
        "FCGR3A_monocytes": ["FCGR3A", "LST1", "IFITM3", "LGALS3"],
        "Dendritic_cells": ["FCER1A", "CST3", "CD74", "HLA-DRA"],
        "Megakaryocytes": ["PPBP", "PF4", "NRGN"],
    },
    "pancreas": {
        "Alpha": ["GCG", "TTR", "LOXL4"],
        "Beta": ["INS", "IAPP", "PCSK1", "MAFA"],
        "Delta": ["SST", "RBP4", "GHSR"],
        "Gamma_PP": ["PPY", "PAX6", "CARTPT"],
        "Acinar": ["PRSS1", "REG1A", "AMY2A"],
        "Ductal": ["KRT19", "KRT8", "KRT18", "SPP1"],
        "Endothelial": ["KDR", "ESAM", "VWF", "EMCN"],
        "Stellate": ["COL1A1", "COL3A1", "DCN", "COL1A2"],
        "Immune": ["PTPRC", "LYZ", "HLA-DRA"],
    },
}
marker_panel = MARKERS[DATASET]
marker_panel


# 6. AI Figure Coach — use this protocol exactly

Paste the **Master prompt** into a separate AI chat once, then use the **Stage prompts** in order. A student runs the returned code in a new Colab code cell, inspects its single artifact, records an interpretation, and only then sends the next stage prompt.

## Master prompt — paste once

```text
You are a cautious single-cell RNA-seq teaching assistant for Studio 2. You are coaching a student in a Google Colab notebook. You must work in a strict one-stage-at-a-time mode.

Data privacy: the dataset is public. Do not ask for names, files, credentials, or protected/unpublished data. Do not fabricate results, code execution, marker evidence, citations, or CellTypist model properties.

Notebook data contract:
- canonical AnnData object: adata
- baseline cluster labels: adata.obs["leiden"]
- selected annotation labels: adata.obs["annotation_cluster"]
- selected key: ANNOTATION_CLUSTER_KEY; other candidates are listed in RESOLUTION_KEY_BY_VALUE
- UMAP: adata.obsm["X_umap"]
- adata.X is scaled highly variable genes; use adata.raw for marker expression and CellTypist input
- adata.raw contains log1p, CP10k values across all filtered genes
- raw filtered counts are in adata.layers["counts"] only for the current HVG subset
- variables DATASET, MODEL_NAME, marker_panel, SEED, and OUTPUT_DIR already exist
- use Path(OUTPUT_DIR) for artifacts and never overwrite adata.raw

Rules:
1. At each stage, produce only the exact deliverable requested. For a figure stage, return exactly one self-contained Python code cell and create exactly one new figure artifact. A multi-panel figure is allowed only when the prompt asks for it.
2. Code must start with '# STAGE n', use the existing variables, state its required imports, and save a PNG in OUTPUT_DIR. If simple, also save a PDF. Do not rerun preprocessing, PCA, neighbors, UMAP, Leiden, or CellTypist unless the stage explicitly asks.
3. Never assign a biological label solely from a cluster number, UMAP location, one gene, or a CellTypist prediction. State what additional evidence is needed. Never treat CellTypist as ground truth.
4. Before suggesting a parameter change, give ranked hypotheses, a discriminating diagnostic, the smallest reversible change, and what result would retain, narrow, or withdraw the claim.
5. At the end of every response, stop. Ask the student to run the cell, inspect the artifact, and send the next numbered stage. Do not anticipate, generate, or describe a later stage.
6. If a requested prerequisite object is absent, return a short diagnostic-only cell or explain what is missing; do not improvise a substitute analysis.

Acknowledge these rules in two sentences only. Do not generate any analysis code until I send STAGE 0A.
```

## Verification log

Create a small markdown record under your figures. For every AI interaction, record: prompt purpose; accepted/rejected suggestion; exact code run; diagnostic rerun; figure/table path; interpretation; and whether your claim was retained, narrowed, or withdrawn. This makes AI assistance auditable rather than invisible.


## Stage prompts — send **only one** after the master prompt

### STAGE 0A — Compare Leiden resolutions

```text
STAGE 0A. The existing variables LEIDEN_RESOLUTIONS, RESOLUTION_KEY_BY_VALUE, and resolution_summary were computed on one fixed neighbor graph and UMAP. Create exactly one multi-panel UMAP figure with one panel per candidate cluster key, colored by its cluster labels. Print resolution_summary. Do not assign biological labels, change the graph, or recommend a preferred resolution. Save it as outputs/stage0a_leiden_resolution_comparison.png. Then wait.
```

### STAGE 0B — Select annotation granularity

```text
STAGE 0B. I am considering ANNOTATION_CLUSTER_KEY = <<PASTE CANDIDATE KEY>>. Here is my short rationale based on the resolution comparison, cluster sizes, and marker coherence:
<<PASTE 3–5 SENTENCES>>

Audit the rationale only. State the strongest supporting evidence, one concrete risk of over- or under-clustering, and one marker-based diagnostic I should inspect before annotating. Do not choose a resolution for me and do not write code. Then wait.
```

After the audit, edit ANNOTATION_CLUSTER_KEY in the resolution cell, rerun that cell, and continue.

### STAGE 1 — Cluster overview

```text
STAGE 1. Create exactly one UMAP figure colored by adata.obs["annotation_cluster"]. Put cluster IDs on the plot if labels remain readable. Do not assign biological labels or change clustering. Save it as outputs/stage1_annotation_cluster_umap.png. Then wait.
```

### STAGE 2 — Cluster-specific marker evidence

```text
STAGE 2. First compute rank_genes_groups on adata using groupby="annotation_cluster", method="wilcoxon", use_raw=True, and key_added="rank_genes_annotation_cluster". Then create exactly one figure that displays the top five ranked genes per cluster. Use a visualization appropriate for Scanpy ranked genes. Save it as outputs/stage2_ranked_markers.png. Do not manually annotate or run CellTypist. Include one sentence warning that cell-level p-values are not replicate-level inference. Then wait.
```

### STAGE 3 — Curated marker-panel overview

```text
STAGE 3. Use the existing marker_panel dictionary and adata.raw expression to create exactly one dot plot grouped by "annotation_cluster". Make missing markers safe to skip and print the retained versus absent markers. Use a gene-wise display scale only if you state that it supports pattern comparison, not cross-gene magnitude comparison. Save it as outputs/stage3_marker_dotplot.png. Do not assign labels. Then wait.
```

### STAGE 4 — Resolve one ambiguous cluster

```text
STAGE 4. I need to inspect cluster <<CLUSTER_ID>>. Based only on these candidate identities: <<CANDIDATE_LABELS>>, make exactly one multi-panel UMAP expression figure for 3–6 discriminating genes. Use adata.raw expression, do not choose a label, and state why coherent multi-gene evidence matters. Save it as outputs/stage4_cluster_<<CLUSTER_ID>>_genes.png. Then wait.
```

### STAGE 5 — Manual annotation checkpoint

```text
STAGE 5. Here is my evidence table with each cluster, positive markers, contradictory/absent markers, and tentative label:
<<PASTE YOUR TABLE>>

Audit it without changing my conclusions. For each cluster, separate direct observation from tentative inference, name one failure mode, and identify any label that should be "Review" rather than forced. Do not produce plotting code. Then wait.
```

### STAGE 6 — Enter student-approved manual labels

```text
STAGE 6. I have finalized this student-approved dictionary mapping selected annotation-cluster IDs to manual labels:
<<PASTE A PYTHON DICTIONARY, e.g. {"0": "...", "1": "..."}>>

Return exactly one code cell that validates every observed annotation_cluster is mapped exactly once, creates adata.obs["manual_cell_type"] as a categorical column, and makes exactly one UMAP colored by manual_cell_type. It must fail clearly on missing or extra cluster IDs. Save it as outputs/stage6_manual_labels_umap.png. Do not alter clustering or CellTypist data. Then wait.
```

### STAGE 7 — Model scope check and CellTypist transfer

```text
STAGE 7. Verify whether MODEL_NAME is suitable for the selected DATASET in organism, tissue, and expected label granularity. If the scope is not suitable, stop without annotation and explain the mismatch.

If it is suitable, return exactly one code cell with no plot. It must: create adata_ct = adata.raw.to_adata(); copy adata.obs and X_umap into adata_ct; download MODEL_NAME; call celltypist.annotate on adata_ct with majority_voting=True and over_clustering="annotation_cluster"; convert results with predictions.to_adata(insert_conf_by="majority_voting", prefix="celltypist_"); and copy the prediction, over-clustering, majority-voting, and confidence columns back to adata.obs. Preserve adata, adata.raw, and existing manual labels. Print a concise label-frequency table and stop.
```

### STAGE 8 — CellTypist majority-voting view

```text
STAGE 8. Confirm that adata.obs contains celltypist_majority_voting. Create exactly one UMAP colored by this column. Title it so it clearly says this is CellTypist majority voting over the supplied selected annotation clusters, not ground truth. Save it as outputs/stage8_celltypist_majority_umap.png. Do not merge labels, change the model, or make a manual-vs-automatic conclusion. Then wait.
```

### STAGE 9 — Compare manual and automated labels

```text
STAGE 9. Confirm that manual_cell_type and celltypist_majority_voting exist. Create a row-normalized cross-tabulation of manual labels versus CellTypist majority-voting labels and make exactly one heatmap from it. Print the unnormalized counts too. Save the figure as outputs/stage9_manual_vs_celltypist_heatmap.png and the count/normalized tables as CSV files. Do not automatically harmonize label names or declare a winner. Then wait.
```

### STAGE 10 — Agreement / review adjudication

```text
STAGE 10. I will provide two explicit label-to-broad-lineage dictionaries, one for manual_cell_type and one for celltypist_majority_voting:
manual_to_broad = <<PASTE DICTIONARY>>
celltypist_to_broad = <<PASTE DICTIONARY>>

Return exactly one code cell that validates full coverage of observed labels, creates manual_broad_label and celltypist_broad_label, and sets consensus_label to the shared broad label only when they agree; otherwise use "Review". Create exactly one UMAP colored by consensus_label and save it as outputs/stage10_consensus_or_review_umap.png. Explain in a code comment that two sources do not constitute a statistical majority vote: this is an agreement/review rule. Do not overwrite either original annotation. Then wait.
```

### STAGE 11 — Claim audit

```text
STAGE 11. Here is my proposed conclusion and the paths to my marker, manual-label, CellTypist, comparison, and consensus artifacts:
<<PASTE YOUR 3–5 SENTENCE CONCLUSION AND PATHS>>

Do not write new code. Separate direct observations, supported inference, limitations, and the next validation test. Flag any causal language, claim of ground truth, or unsupported citation. Then wait.
```


## 7. Interpretation guardrails

1. **Resolution choice:** The Leiden resolution controls partition granularity; it does not reveal the true number of cell types. Select a resolution from cluster size, multi-gene marker coherence, and the risks of both merging and splitting, then retain the comparison as evidence.
2. **Manual annotation:** Record a label only when multiple markers support it and competing identities are considered. A continuous UMAP and cluster number do not establish a cell type.
3. **Marker statistics:** `rank_genes_groups` is useful for characterizing clusters. Its cell-level significance is not a biological replicate-level test. Do not use it to claim condition effects.
4. **CellTypist:** This is label transfer from a particular reference. A model can be technically runnable and biologically mismatched. Inspect its tissue, species, disease state, label hierarchy, and gene overlap.
5. **Majority voting:** Here, CellTypist uses the selected annotation clusters to refine its own cell-wise predictions within local groups. It is not a vote between independent experimental measurements.
6. **Manual vs CellTypist:** Do not force names to agree. A manual/CellTypist match is a useful working consensus; a disagreement is an evidence gap to investigate with more markers, QC, doublet checks, or an appropriate external reference.
7. **Pancreas caution:** Ambient endocrine transcripts, especially insulin, can appear outside beta cells. Require concordant marker evidence and inspect cluster context.

**Bounded sensitivity extension:** After submitting the baseline, you may evaluate one additional resolution outside the provided set or one other reversible graph parameter. Preserve the original results, change one parameter only, repeat the comparison, and state whether the original annotation claim is retained, narrowed, or withdrawn. Do not select a setting merely because it produces a preferred label composition.

## 8. Submission checklist

- [ ] Dataset, URL/loader, package versions, seed, and baseline parameters are recorded.
- [ ] I retained the raw-to-log1p data contract required by CellTypist.
- [ ] I compared the supplied Leiden resolutions on the fixed graph, selected `annotation_cluster`, and documented why its granularity is defensible.
- [ ] I generated and inspected a cluster UMAP, ranked markers, a marker dot plot, and at least one ambiguity-resolving gene-expression plot.
- [ ] I documented multi-marker evidence and a limitation for every manual label.
- [ ] I checked CellTypist model scope before label transfer.
- [ ] I ran CellTypist with `majority_voting=True` and my selected annotation clusters supplied as `over_clustering`.
- [ ] I exported the manual-vs-CellTypist comparison and retained disagreements as `Review` rather than overwriting labels.
- [ ] I kept an AI verification log showing what was accepted, rejected, rerun, and revised.

## Sources

1. Scanpy’s current preprocessing/clustering tutorial documents QC metrics, count-depth normalization, log transformation, HVG selection, graph construction, Leiden clustering, marker panels, and cluster marker ranking: https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html
2. Scanpy’s `pbmc3k()` loader documents the 2,700-cell healthy-donor PBMC dataset: https://scanpy.readthedocs.io/en/stable/api/scanpy.datasets.pbmc3k.html
3. The scvi-tools pancreas tutorial documents the scIB pancreas AnnData source, dimensions, and `counts` layer: https://docs.scvi-tools.org/en/1.2.2/tutorials/notebooks/scrna/scarches_scvi_tools.html
4. CellTypist documents its AnnData requirement for log1p normalized 10,000-count expression, pretrained model download, prediction, and majority voting: https://github.com/Teichlab/celltypist
5. CellTypist’s API documents the `annotate` parameters including `majority_voting` and `over_clustering`: https://celltypist.readthedocs.io/en/latest/_modules/celltypist/annotate.html
6. Scanpy’s marker-ranking API notes that `rank_genes_groups` expects logarithmized data and warns that cell-level comparisons can inflate p-values: https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.rank_genes_groups.html

**Version note:** package and model catalogues evolve. At the start of each course offering, run `models.models_description()` in Colab, record the current model details, and review the model’s source before students use it.
